# Public EVBattery reproduction bootstrap

No author-produced input ZIP is required for this public-fleet route. Obtain the original three archives from https://doi.org/10.17632/mcsh4hnb8b.1 and place them anywhere under My Drive. All preparation/training here is remote. Use an A100 for full training. Pin GIT_REF to the publication commit/tag for a frozen reproduction; the initial main branch is not a Zenodo release.

This reproduces the Stage 12 sampling/Stage 13 persistence-anchored fleet route and optional adapted recent controls. It does not silently rerun Nissan, the Stage 15 selective policy, or unsupported thermal targets. Cached preparation and completed full stages are reused; the historical Stage 13 loop does not provide epoch-by-epoch resume.

In [ ]:
# BLOCK 1 — Protocol and remote storage
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, queue, shutil, subprocess, sys, threading, time
GIT_REF='main'  # replace with the exact release commit/tag when published
DRIVE_SEARCH_ROOT='/content/drive/MyDrive'
RUN_ID='dbpatchkan_public_fleet_v0_1_0'
SEEDS=[42,52,62]
FOLDS=5
EPOCHS=60
BATCH_SIZE=512
RUN_RECENT_METHODS=False  # True additionally runs Stage 13B and post-hoc 13C
RUN_SMOKE_TESTS=True
def update(message):
    print(f'[{datetime.now(timezone.utc).strftime("%H:%M:%S")} UTC] {message}',flush=True)
update('BLOCK 1 complete. All computation runs on Colab, not your desktop.')

In [ ]:
# BLOCK 2 — Mount Drive, obtain code, and install on Colab
from google.colab import drive
drive.mount('/content/drive')
PROJECT=Path('/content/DB-PatchKAN_public')
if not PROJECT.exists():
    subprocess.run(['git','clone','https://github.com/Simhaatt/DB-PatchKAN.git',str(PROJECT)],check=True)
else:
    origin=subprocess.check_output(['git','-C',str(PROJECT),'remote','get-url','origin'],text=True).strip()
    assert origin=='https://github.com/Simhaatt/DB-PatchKAN.git','Unexpected existing repository'
    assert not subprocess.check_output(['git','-C',str(PROJECT),'status','--porcelain'],text=True).strip(),'Existing clone has edits; use a fresh Colab runtime or preserve them separately'
    subprocess.run(['git','-C',str(PROJECT),'fetch','origin'],check=True)
ref='origin/main' if GIT_REF=='main' else GIT_REF
subprocess.run(['git','-C',str(PROJECT),'checkout','--detach',ref],check=True)
COMMIT=subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'],text=True).strip()
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(PROJECT)+'[dev,tuning,repro]'],check=True)
sys.path.insert(0,str(PROJECT/'src'))
os.environ['PYTHONPATH']=str(PROJECT/'src')+os.pathsep+os.environ.get('PYTHONPATH','')
import torch
assert torch.cuda.is_available(),'Choose Runtime > Change runtime type > GPU; use A100 for the full run'
RUN=Path('/content/drive/MyDrive/DBPatchKAN_public_reproduction')/RUN_ID
RUN.mkdir(parents=True,exist_ok=True)
update(f'BLOCK 2 complete: commit={COMMIT}, GPU={torch.cuda.get_device_name(0)}, run={RUN}')

In [ ]:
# BLOCK 3 — Find and verify official raw archives; freeze this run's protocol
expected={
 'battery_dataset1.tar.gz':(812110847,'BF56F38EF1F0C96BB184F085F261F72092D16366C5EFB7264DF55E68FBA16BE5'),
 'battery_dataset2.tar.gz':(418251471,'265E3A7B6CEDCD8A6C1844C20C9E69F0DE7DCDEE07F10816C29479791366107F'),
 'battery_dataset3.tar.gz':(177847087,'0B258606D79D6392C0F8C76E74B1F544650AB7FBF29E758F078C2C6AB74630AD')}
def sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(4*2**20),b''):digest.update(chunk)
    return digest.hexdigest().upper()
found={name:[] for name in expected}
last=time.monotonic();folders=0
for folder,dirs,names in os.walk(DRIVE_SEARCH_ROOT):
    folders+=1
    for name in names:
        if name in found:found[name].append(Path(folder)/name)
    if time.monotonic()-last>20:
        update(f'Drive search: {folders:,} folders checked');last=time.monotonic()
archives=[]
for name,(size,wanted) in expected.items():
    valid=None
    for path in sorted(found[name]):
        if path.stat().st_size!=size:continue
        update(f'Hashing {name}: {size/2**20:.1f} MB remotely')
        if sha256(path)==wanted:valid=path;break
    assert valid is not None,f'Missing or different source release: {name}; obtain version 1 from the original DOI and finish the Drive upload'
    archives.append(valid);update(f'Verified {valid}')
PROTOCOL={'commit':COMMIT,'folds':FOLDS,'seeds':SEEDS,'epochs':EPOCHS,'batch_size':BATCH_SIZE,
          'max_members_per_archive':200000,'max_records_per_vehicle':64,'sampling_seed':42,
          'raw_sha256':{n:v[1] for n,v in expected.items()},'recent_methods':RUN_RECENT_METHODS}
protocol_file=RUN/'REPRODUCTION_PROTOCOL.json'
if protocol_file.exists():
    assert json.loads(protocol_file.read_text())==PROTOCOL,'Protocol changed: choose a new RUN_ID; do not mix results'
else:protocol_file.write_text(json.dumps(PROTOCOL,indent=2))
(RUN/'runtime.json').write_text(json.dumps({'python':sys.version,'torch':torch.__version__,
 'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'commit':COMMIT},indent=2))
with (RUN/'requirements-freeze.txt').open('w') as handle:
    subprocess.run([sys.executable,'-m','pip','freeze'],stdout=handle,check=True)
update('BLOCK 3 complete: source hashes and protocol fixed')

In [ ]:
# BLOCK 4 — Live logs/heartbeats and optional remote synthetic checks
def run(command,label):
    update(f'START {label}');started=time.monotonic()
    logfile=RUN/(label.replace(' ','_')+'.log')
    process=subprocess.Popen(command,cwd=PROJECT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    messages=queue.Queue()
    def reader():
        try:
            for line in process.stdout:messages.put(line)
        finally:messages.put(None)
    threading.Thread(target=reader,daemon=True).start()
    ended=False;last=started
    with logfile.open('a',encoding='utf-8') as log:
        while not ended:
            try:
                line=messages.get(timeout=1)
                if line is None:ended=True
                else:print(line,end='',flush=True);log.write(line);log.flush()
            except queue.Empty:pass
            if time.monotonic()-last>=30:
                update(f'{label}: elapsed={(time.monotonic()-started)/60:.1f} min');last=time.monotonic()
    code=process.wait()
    assert code==0,f'{label} failed (exit={code}); inspect {logfile}; do not treat partial outputs as completed'
    update(f'DONE {label}: {(time.monotonic()-started)/60:.1f} min')
run([sys.executable,'scripts/verify_repository.py'],'repository integrity')
if RUN_SMOKE_TESTS:
    run([sys.executable,'-m','pytest','tests/test_models.py','tests/test_stage13_persistence_forecast.py','tests/test_stage15_remote.py'],'synthetic tests')
update('BLOCK 4 complete')

In [ ]:
# BLOCK 5 — Prepare deterministic charging sample and mileage transitions
SAMPLE=RUN/'evbattery_stage12_sample.npz'
CACHE=RUN/'evbattery_stage13_transitions.npz'
if not SAMPLE.exists():
    run([sys.executable,'-m','dbpatchkan.analysis.stage12_evbattery','prepare','--archives',*[str(p) for p in archives],
         '--output',str(SAMPLE),'--max-members-per-archive','200000','--max-records-per-vehicle','64','--seed','42'],'charging sample preparation')
else:update('Reusing cached charging sample under the unchanged protocol')
if not CACHE.exists():
    run([sys.executable,'-m','dbpatchkan.analysis.stage13_persistence_forecast','build','--source-cache',str(SAMPLE),
         '--output',str(CACHE)],'transition cache preparation')
else:update('Reusing cached transition inputs under the unchanged protocol')
from dbpatchkan.analysis.stage13_persistence_forecast import load_transition_cache
loaded=load_transition_cache(CACHE)
table=loaded['table']
update(f'Prepared {len(table):,} transitions across {table.vehicle_key.nunique()} vehicles')
assert len(table)==9714 and table.vehicle_key.nunique()==191,'Cohort differs from the recorded experiment; inspect the sampling audit before comparison'
del loaded,table
update('BLOCK 5 complete')

In [ ]:
# BLOCK 6 — Main fleet forecasts, with optional matched recent controls
S13=RUN/'stage13';S13B=RUN/'stage13b';S13C=RUN/'stage13c'
common=['--folds',str(FOLDS),'--seeds',*[str(s) for s in SEEDS],'--epochs',str(EPOCHS),'--batch-size',str(BATCH_SIZE)]
if not (S13/'STAGE13_AUDIT.json').exists():
    run([sys.executable,'-m','dbpatchkan.analysis.stage13_persistence_forecast','validate','--cache',str(CACHE),
         '--output-dir',str(S13),*common],'Stage 13 fleet forecasts')
else:update('Stage 13 completed outputs exist; skipping training')
if RUN_RECENT_METHODS:
    if not (S13B/'STAGE13B_AUDIT.json').exists():
        run([sys.executable,'-m','dbpatchkan.analysis.stage13b_recent_methods','--cache',str(CACHE),
             '--stage13-results',str(S13),'--output-dir',str(S13B),*common],'Stage 13B adapted controls')
    if not (S13C/'STAGE13C_AUDIT.json').exists():
        run([sys.executable,'-m','dbpatchkan.analysis.stage13c_timepro','--cache',str(CACHE),
             '--frozen-stage13b',str(S13B),'--output-dir',str(S13C),*common],'Stage 13C posthoc TimePro')
import pandas as pd
display(pd.read_csv(S13/'stage13_seed_ensemble_metrics.csv'))
display(pd.read_csv(S13/'stage13_paired_vehicle_bootstrap.csv'))
update('BLOCK 6 complete. Save scope and all comparator results, including negative findings.')

In [ ]:
# BLOCK 7 — Save to Drive and automatically download models/results ZIP
from google.colab import files
archive_base=Path('/content')/(RUN_ID+'_MODELS_RESULTS')
archive=Path(shutil.make_archive(str(archive_base),'zip',root_dir=RUN))
drive_copy=RUN.parent/archive.name
shutil.copy2(archive,drive_copy)
update(f'Archive ready: {archive.stat().st_size/2**20:.1f} MB; Drive backup={drive_copy}')
files.download(str(archive))
update('BLOCK 7 complete. No result was published to GitHub or Zenodo by this notebook.')